# 03 · Variables y baselines

**Qué hace:** muestra `rc_pronostico` en acción con un plato real de Stuttgart: la tabla de variables de un punto de corte, el tratamiento de los días cerrados, los baselines frente a lo vendido y la comprobación de que no hay fuga del futuro.

**Decisiones que aplica:** D-07 (variables), D-09 (baselines), D-05 (cerrado = dato faltante), D-10 (tramo de arranque), D-13 (objetivos excluidos).

**Por qué importa:** este mismo código, y no una copia, arma las variables en el servicio. Si el entrenamiento y el servicio calcularan algo distinto, el modelo recibiría en producción datos que nunca vio (*training-serving skew*).

**Necesita:** `salidas/stuttgart/` del cuaderno 01. **Deja:** `salidas/stuttgart/filas_entrenamiento_demo.parquet` (solo para inspección).

In [1]:
# Parámetros
import json
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "cuadernos" else Path.cwd()
SALIDA = RAIZ / "salidas" / "stuttgart"
PLATO = "STEAK"
CORTE = "2014-11-07"          # fin del entrenamiento inicial (D-10); nada posterior se usa aquí
CORTE_NAVIDAD = "2014-12-22"  # para ver los días cerrados de 2014

In [2]:
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
from rc_pronostico.variables import (LISTA_VARIABLES, VERSION_VARIABLES, DIAS_ARRANQUE,
                                     codigos_producto, filas_entrenamiento)
from rc_pronostico.baselines import baselines

historial = pd.read_parquet(SALIDA / "historial.parquet")
calendario = json.loads((SALIDA / "calendario.json").read_text(encoding="utf-8"))
codigos = codigos_producto(historial)
print("version_variables:", VERSION_VARIABLES, "| variables:", len(LISTA_VARIABLES))
print(codigos)

version_variables: 1 | variables: 21
{'CALAMARI': 0, 'FISCH': 1, 'GARNELEN': 2, 'HAEHNCHEN': 3, 'KOEFTE': 4, 'LAMM': 5, 'STEAK': 6}


## 1. Las variables de un punto de corte
Para el corte `d`, plato y distancia `h`, se pronostica el día `t = d + h`. Abajo, el corte `d = 2014-11-07` (viernes) de STEAK para `h` de 1 a 8: cada fila es un día objetivo distinto con sus variables.

In [3]:
filas = filas_entrenamiento(historial, calendario, "2014-11-15", codigos)   # objetivos hasta el 15/11
una = filas[(filas.producto_id == PLATO) & (filas.fecha_corte == CORTE)].sort_values("h")
una[["fecha_objetivo", "y", *LISTA_VARIABLES]].set_index("fecha_objetivo").T

fecha_objetivo,2014-11-08,2014-11-09,2014-11-10,2014-11-11,2014-11-12,2014-11-13,2014-11-14,2014-11-15
y,62.000000,23.000000,15.000000,22.000000,19.000000,15.000000,26.000000,56.000000
dia_semana,5.000000,6.000000,0.000000,1.000000,2.000000,3.000000,4.000000,5.000000
dia_mes,8.000000,9.000000,10.000000,11.000000,12.000000,13.000000,14.000000,15.000000
mes,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000,11.000000
es_feriado,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
vispera_feriado,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
pos_feriado,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
h,1.000000,2.000000,3.000000,4.000000,5.000000,6.000000,7.000000,8.000000
producto,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000,6.000000
lag0,26.000000,26.000000,26.000000,26.000000,26.000000,26.000000,26.000000,26.000000


Se lee así: `lag0` a `lag6` son las ventas de `d` y los 6 días anteriores; `media_7`, `media_28` y `desv_28` resumen el nivel reciente; `ultimo_mismo_dia` y `media_mismo_dia_4` miran el mismo día de la semana que el objetivo (con `h = 8` el último disponible es 14 días antes, no 7); `tendencia` es `media_7 / media_28`. El año no entra (D-12).

## 2. Días cerrados: dato faltante, no cero (D-05)
Cierres de Stuttgart en diciembre de 2014: 24, 25, 26 y 31. Con corte el 22/12 y STEAK:

In [4]:
fil_nav = filas_entrenamiento(historial, calendario, "2015-01-05", codigos)
nav = fil_nav[(fil_nav.producto_id == PLATO) & (fil_nav.fecha_corte >= "2014-12-22") & (fil_nav.fecha_corte <= "2014-12-27") & (fil_nav.h == 1)]
nav[["fecha_corte", "fecha_objetivo", "y", "lag0", "lag1", "lag2", "media_7", "ultimo_mismo_dia"]]

,fecha_corte,fecha_objetivo,y,lag0,lag1,lag2,media_7,ultimo_mismo_dia
23001,2014-12-22,2014-12-23,14.0,32.0,5.0,38.0,23.428571,19.0
23141,2014-12-26,2014-12-27,25.0,NaN,NaN,NaN,22.250000,38.0
23190,2014-12-27,2014-12-28,9.0,25.0,NaN,NaN,19.000000,5.0


Se observa que **no hay fila con objetivo el 24, 25 ni 26** (no son objetivo). Con corte el 26/12 (cerrado), `lag0`, `lag1` y `lag2` son `NaN` (26, 25 y 24), no 0; con corte el 27/12, `lag1` es `NaN`. La media de 7 días se calcula con los días abiertos, así que los cierres no arrastran el nivel hacia abajo.

In [5]:
cerrados = pd.DatetimeIndex(calendario["dias_cerrados"])
print("Objetivos en días cerrados:", int(fil_nav.fecha_objetivo.isin(cerrados).sum()))
hist_p = historial[historial.producto_id == PLATO].set_index("fecha").unidades
print("El historial trae 0 el 25/12/2014 para ser continuo:", hist_p["2014-12-25"], "| pero como variable es NaN:", bool(fil_nav[(fil_nav.producto_id == PLATO) & (fil_nav.fecha_corte == "2014-12-26") & (fil_nav.h == 1)].lag1.isna().iloc[0]))

Objetivos en días cerrados: 0
El historial trae 0 el 25/12/2014 para ser continuo: 0 | pero como variable es NaN: True


## 3. Baselines frente a lo vendido (D-09)
Solo con objetivos del entrenamiento inicial (hasta 2014-11-07). La prueba no se toca aquí.

In [6]:
ent = filas_entrenamiento(historial, calendario, CORTE, codigos)
bl = baselines(historial, calendario, ent)
ent = ent.join(bl)
ent["ae_ingenuo"] = (ent.y - ent.ingenuo).abs()
ent["ae_media4"] = (ent.y - ent.media_mismo_dia).abs()
ent["cubre_p90"] = ent.y <= ent.p90_mismo_dia
tabla = ent.groupby("producto_id").agg(
    n=("y", "size"), MAE_ingenuo=("ae_ingenuo", "mean"), MAE_media4=("ae_media4", "mean"),
    cobertura_P90_baseline=("cubre_p90", "mean")).round(3)
tabla

,n,MAE_ingenuo,MAE_media4,cobertura_P90_baseline
producto_id,,,,
CALAMARI,2912,2.766,2.198,0.843
FISCH,2912,2.721,2.134,0.844
GARNELEN,2912,4.315,3.394,0.838
HAEHNCHEN,2912,8.418,6.897,0.834
KOEFTE,2912,7.172,5.827,0.809
LAMM,2912,8.482,7.166,0.827
STEAK,2912,6.758,5.575,0.860


In [7]:
# Un plato, una semana: lo vendido y lo que dicen los baselines (h = 2, corte semanal)
ejemplo = ent[(ent.producto_id == PLATO) & (ent.h == 2)].tail(10)
ejemplo[["fecha_corte", "fecha_objetivo", "y", "ingenuo", "media_mismo_dia", "p90_mismo_dia"]]

,fecha_corte,fecha_objetivo,y,ingenuo,media_mismo_dia,p90_mismo_dia
19977,2014-10-27,2014-10-29,12.0,18.0,15.25,24.0
20033,2014-10-28,2014-10-30,19.0,21.0,18.25,22.2
20089,2014-10-29,2014-10-31,24.0,29.0,22.00,26.2
20145,2014-10-30,2014-11-01,32.0,49.0,35.75,39.9
20201,2014-10-31,2014-11-02,13.0,16.0,15.25,19.9
20250,2014-11-01,2014-11-03,24.0,14.0,16.25,21.5
20292,2014-11-02,2014-11-04,21.0,18.0,15.75,20.0
20327,2014-11-03,2014-11-05,16.0,12.0,14.00,24.0
20355,2014-11-04,2014-11-06,24.0,19.0,19.00,22.2
20376,2014-11-05,2014-11-07,26.0,24.0,21.75,26.2


## 4. Comprobación de fuga a la vista
Se destruye todo lo posterior al corte (se reemplaza por 99 999 unidades) y se vuelven a calcular las variables y los baselines. Si algo mirara el futuro, cambiaría. La misma prueba corre en `pytest` (`test_sin_fuga_ignora_datos_posteriores_al_corte`).

In [8]:
corte = pd.Timestamp("2014-09-12")
original = filas_entrenamiento(historial, calendario, corte, codigos)

alterado = historial.copy()
alterado.loc[alterado.fecha > corte, "unidades"] = 99_999
despues = filas_entrenamiento(alterado, calendario, corte, codigos)

identicas = original.equals(despues)
bl_a = baselines(historial, calendario, original)
bl_b = baselines(alterado, calendario, original)
print("Variables idénticas tras alterar el futuro:", identicas)
print("Baselines idénticos tras alterar el futuro:", bl_a.equals(bl_b))
print("Último objetivo:", original.fecha_objetivo.max().date(), "| corte:", corte.date())

Variables idénticas tras alterar el futuro: True
Baselines idénticos tras alterar el futuro: True
Último objetivo: 2014-09-12 | corte: 2014-09-12


## 5. Qué filas entran al entrenamiento (D-10, D-13)

In [9]:
ent = filas_entrenamiento(historial, calendario, CORTE, codigos)
primera = historial.fecha.min()
resumen = pd.Series({
    "filas": len(ent), "platos": ent.producto_id.nunique(),
    "primer objetivo": ent.fecha_objetivo.min().date(), "último objetivo": ent.fecha_objetivo.max().date(),
    "primer día permitido (arranque)": (primera + pd.Timedelta(days=DIAS_ARRANQUE)).date(),
    "objetivos en días cerrados": int(ent.fecha_objetivo.isin(cerrados).sum()),
    "h mínimo": ent.h.min(), "h máximo": ent.h.max()})
resumen

filas                                   20384
platos                                      7
primer objetivo                    2013-11-08
último objetivo                    2014-11-07
primer día permitido (arranque)    2013-11-08
objetivos en días cerrados                  0
h mínimo                                    1
h máximo                                    8
dtype: object

In [10]:
ent.to_parquet(SALIDA / "filas_entrenamiento_demo.parquet", index=False)
faltantes = ent[LISTA_VARIABLES].isna().mean().round(3)
faltantes[faltantes > 0]

lag0    0.003
lag1    0.003
lag2    0.003
lag3    0.003
lag4    0.003
lag5    0.003
lag6    0.003
dtype: float64

## Verificaciones

In [11]:
assert identicas, "las variables cambiaron al alterar datos posteriores al corte: hay fuga"
assert bl_a.equals(bl_b), "los baselines cambiaron al alterar datos posteriores al corte: hay fuga"
assert original.fecha_objetivo.max() <= corte
assert ent.fecha_objetivo.max() <= pd.Timestamp(CORTE)
assert ent.fecha_objetivo.min() >= primera + pd.Timedelta(days=DIAS_ARRANQUE)
assert int(ent.fecha_objetivo.isin(cerrados).sum()) == 0
assert int(fil_nav.fecha_objetivo.isin(cerrados).sum()) == 0
assert (ent.fecha_corte < ent.fecha_objetivo).all() and ent.h.between(1, 8).all()
assert ent.producto_id.nunique() == 7
assert list(ent[LISTA_VARIABLES].columns) == LISTA_VARIABLES
assert (baselines(historial, calendario, ent).ingenuo == ent.ultimo_mismo_dia).all()
print("OK · variables y baselines sin fuga")

OK · variables y baselines sin fuga
